# Road Damage Detection - Part 4: Hyperparameter tuning on Google Colab (YOLOv10s)
Baseline: YOLOv10s + SGD, 40 epochs, imgsz 640, lr0 0.01 (test mAP50-95 = 0.384).
Tuned runs: **T1** 80 epochs (~1.9 h), **T2** imgsz 800 (~1.5 h), **T3** lr0 0.005 (~0.9 h). Run them in this order; stop whenever you want - step 3 evaluates whatever has finished.

Before you start: *Runtime -> Change runtime type -> T4 GPU*. Keep this tab open while training.
Everything is saved to your Google Drive, so if Colab disconnects: *Reconnect*, re-run all cells from the top, and unfinished runs resume automatically.

## 0. Check GPU, mount Google Drive, install

In [1]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')          # click Allow when asked
!pip install -q ultralytics roboflow

GPU: Tesla T4
Mounted at /content/drive


## 1. Download the dataset (asks for your Roboflow API key)

In [2]:
import gc, yaml, shutil
import numpy as np, pandas as pd
from pathlib import Path
from getpass import getpass
from roboflow import Roboflow
from ultralytics import YOLO

DATA_DIR = Path('/content/road-damage')
if not (DATA_DIR / 'data.yaml').exists():
    try:
        from google.colab import userdata
        key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        key = getpass('Paste your Roboflow API key and press Enter: ')
    rf = Roboflow(api_key=key)
    rf.workspace('mazen-ayad').project('road-damage-l1ju7-g2jni').version(1).download('yolov8', location=str(DATA_DIR))

cfg = yaml.safe_load(open(DATA_DIR / 'data.yaml'))
cfg.update(path=str(DATA_DIR), train='train/images', val='valid/images', test='test/images')
yaml.safe_dump(cfg, open(DATA_DIR / 'data.yaml', 'w'))
DATA_YAML = str(DATA_DIR / 'data.yaml')
print('Dataset ready:', DATA_YAML)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Paste your Roboflow API key and press Enter: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/road-damage in yolov8:: 100%|██████████| 13518/13518 [00:14<00:00, 930.06it/s] 


Dataset ready: /content/road-damage/data.yaml


## 2. Settings and the training helper

In [3]:
RUNS = Path('/content/drive/MyDrive/road_damage_runs_tuned')   # saved on Google Drive
RUNS.mkdir(parents=True, exist_ok=True)

BASE = dict(data=DATA_YAML, optimizer='SGD', lr0=0.01, imgsz=640, batch=16, workers=2, seed=0,
            project=str(RUNS), exist_ok=True, plots=True, device=0)

def train_tuned(name, weights, **overrides):
    """Train one run. Skips finished runs, resumes interrupted ones, never stops the notebook on error."""
    d = RUNS / name
    if (d / 'DONE').exists():
        print('Already done, skipping:', name); return
    gc.collect(); torch.cuda.empty_cache()
    try:
        last = d / 'weights' / 'last.pt'
        if last.exists():
            print('Resuming interrupted run:', name)
            YOLO(str(last)).train(resume=True)
        else:
            YOLO(weights).train(name=name, **{**BASE, **overrides})
        (d / 'DONE').write_text('ok')
    except Exception as e:
        print(f'!!! {name} FAILED: {e!r}')

## Run T1: 80 epochs (about 1.9 h)

In [4]:
train_tuned('v10s_sgd_e80', 'yolov10s.pt', epochs=80, patience=20)

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road-damage/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v10s_sgd_e80, nbs=64, nms=None, opset=None, optimize=False

## Run T2: image size 800 (about 1.5 h)

In [5]:
train_tuned('v10s_sgd_img800', 'yolov10s.pt', epochs=40, patience=10, imgsz=800, batch=12)

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road-damage/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v10s_sgd_img800, nbs=64, nms=None, opset=None, optimize=Fa

## Run T3: lower learning rate 0.005 (about 0.9 h)

In [6]:
train_tuned('v10s_sgd_lr0005', 'yolov10s.pt', epochs=40, patience=10, lr0=0.005)

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/road-damage/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v10s_sgd_lr0005, nbs=64, nms=None, opset=None, optimize=F

## 3. Evaluate every finished run on the TEST set + save results

In [7]:
TEST_IMGS = sorted((DATA_DIR / 'test' / 'images').glob('*'))
rows, per_class = [], {}

for d in sorted(RUNS.glob('v*_*')):
    if not (d / 'DONE').exists():
        continue
    name = d.name
    w = d / 'weights' / 'best.pt'
    imgsz = int(yaml.safe_load(open(d / 'args.yaml'))['imgsz'])
    hist = pd.read_csv(d / 'results.csv'); hist.columns = [c.strip() for c in hist.columns]
    gc.collect(); torch.cuda.empty_cache()

    m = YOLO(str(w)).val(data=DATA_YAML, split='test', imgsz=imgsz, batch=8, conf=0.001, plots=False,
                         project='/content/test_eval', name=name, exist_ok=True, verbose=False, device=0)
    per_class[name] = {m.names[c]: float(m.box.ap50[i]) for i, c in enumerate(m.box.ap_class_index)}

    mdl, t = YOLO(str(w)), []
    for p in TEST_IMGS[:110]:
        r = mdl.predict(str(p), imgsz=imgsz, conf=0.25, device=0, verbose=False)[0]
        t.append(sum(r.speed.values()))
    rows.append(dict(run=name, imgsz=imgsz, epochs_done=len(hist),
                     val_mAP50_95=hist['metrics/mAP50-95(B)'].max(),
                     precision=m.box.mp, recall=m.box.mr, mAP50=m.box.map50, mAP50_95=m.box.map,
                     GPU_FPS_total=1000 / np.mean(t[10:]), size_MB=w.stat().st_size / 1e6))

res = pd.DataFrame(rows).round(3)
# reference row = the earlier baseline (copied from the previous Kaggle test evaluation)
baseline = dict(run='v10s_sgd (baseline, 40 ep)', imgsz=640, epochs_done=40, val_mAP50_95=0.256,
                precision=0.597, recall=0.595, mAP50=0.612, mAP50_95=0.384, GPU_FPS_total=96.7, size_MB=16.5)
res = pd.concat([pd.DataFrame([baseline]), res], ignore_index=True)
res.to_csv(RUNS / 'tuning_results.csv', index=False)
pc = pd.DataFrame(per_class).round(3); pc.to_csv(RUNS / 'tuning_per_class_AP50.csv')
display(res); display(pc)

out = Path('/content/tuned_weights'); out.mkdir(exist_ok=True)
for d in RUNS.glob('v*_*'):
    if (d / 'DONE').exists():
        shutil.copy(d / 'weights' / 'best.pt', out / f'{d.name}.pt')
shutil.make_archive(str(RUNS / 'tuned_weights'), 'zip', out)
print('Saved in your Google Drive folder road_damage_runs_tuned: tuning_results.csv, tuning_per_class_AP50.csv, tuned_weights.zip')

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10s summary (fused): 103 layers, 7,220,709 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 730.5±1546.2 MB/s, size: 249.1 KB)
val: Scanning /content/road-damage/test/labels... 681 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 681/681 662.8it/s 1.0s
val: New cache created: /content/road-damage/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 86/86 7.3it/s 11.8s
                   all        681       2004      0.598      0.599        0.6      0.376
Speed: 0.9ms preprocess, 9.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10s summary (fused): 103 layers, 7,220,709 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2314.4±824.3 MB/s, size: 159.2 KB)
v

,run,imgsz,epochs_done,val_mAP50_95,precision,recall,mAP50,mAP50_95,GPU_FPS_total,size_MB
0,"v10s_sgd (baseline, 40 ep)",640,40,0.256,0.597,0.595,0.612,0.384,96.700,16.500
1,v10s_sgd_e80,640,80,0.253,0.598,0.599,0.600,0.376,69.378,16.526
2,v10s_sgd_img800,800,40,0.251,0.607,0.577,0.610,0.377,46.274,16.544
3,v10s_sgd_lr0005,640,40,0.259,0.600,0.621,0.621,0.390,73.447,16.520


,v10s_sgd_e80,v10s_sgd_img800,v10s_sgd_lr0005
alligator,0.356,0.378,0.361
block,0.868,0.852,0.889
crack,0.789,0.861,0.859
longitudinal,0.223,0.267,0.268
pothole,0.694,0.711,0.693
transverse,0.671,0.592,0.657


Saved in your Google Drive folder road_damage_runs_tuned: tuning_results.csv, tuning_per_class_AP50.csv, tuned_weights.zip
